In [0]:
NOBIL_KEY = dbutils.secrets.get(scope="kv-labb2", key="nobil-api-key")
SEARCH_URL = "https://nobil.no/api/server/search.php"


In [0]:
import os, time, requests
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo

DUMP_URL = "https://nobil.no/api/server/datadump.php"

def nobil_post(url, timeout=300, **params):
    # POST keeps the key out of the URL (and out of exception messages)
    resp = requests.post(url, data={"apikey": NOBIL_KEY, **params}, timeout=timeout)
    resp.raise_for_status()
    return resp

snapshot_date = datetime.now(ZoneInfo("Europe/Stockholm")).date().isoformat()
land_dir   = f"/Volumes/laddstolpar_df/landing/raw/nobil/snapshot_date={snapshot_date}"
stats_path = f"{land_dir}/stats_totals_all_counties.json"
dump_path  = f"{land_dir}/nobil_swe.json"          # written last = completion marker

if os.path.exists(dump_path):
    print(f"SKIP: {dump_path} already exists")
else:
    os.makedirs(land_dir, exist_ok=True)

    # 1. Reconciliation target, fetched in the same run
    stats_resp = nobil_post(SEARCH_URL, timeout=60, apiversion="3", action="search",
                            type="stats_TotalsAllCounties", countrycode="SWE", format="json")
    expected = sum(c["count"] for c in stats_resp.json()["chargerstations"])

    # 2. Full dump for Sweden
    t0 = time.time()
    dump_resp = nobil_post(DUMP_URL, countrycode="SWE", format="json", file="false", norealtime="false", nonimupdate="false")
    secs = time.time() - t0

    # 3. Structural checks only (a failure here means: don't land it)
    dump = dump_resp.json()                      # raises if we got XML or an error text
    stations = dump["chargerstations"]
    assert len(stations) > 0, "empty dump"
    lands = Counter(s["csmd"].get("Land_code") for s in stations)
    assert set(lands) == {"SWE"}, f"unexpected Land_code: {lands}"

    # 4. Land both responses unchanged, dump last
    with open(stats_path, "wb") as f: f.write(stats_resp.content)
    with open(dump_path,  "wb") as f: f.write(dump_resp.content)

    print(f"{len(stations):,} stations in {secs:.1f} s, {len(dump_resp.content)/1e6:.1f} MB")
    print(f"stats (active) = {expected:,}  diff = {len(stations) - expected:+,}")
    print("Station_status:", Counter(s["csmd"].get("Station_status") for s in stations))
    print("Rights:", dump.get("Rights"), "| top-level keys:", list(dump))

Load into bronze

In [0]:
import glob, json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

T = "laddstolpar_df.bronze.nobil_stations"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T} (
  station_id     BIGINT    COMMENT 'csmd.id, extracted for convenience',
  csmd           VARIANT   COMMENT 'station fields, unchanged',
  attr           VARIANT   COMMENT 'attr.st + attr.conn, unchanged',
  _snapshot_date DATE      COMMENT 'from the landing folder name',
  _rights        STRING    COMMENT 'licence string from the response',
  _file          STRING,
  _ingested_at   TIMESTAMP
) COMMENT 'NOBIL datadump (countrycode=SWE), one row per station per snapshot'
""")

files = sorted(glob.glob("/Volumes/laddstolpar_df/landing/raw/nobil/snapshot_date=*/nobil_swe.json"))
done  = {r._file for r in spark.table(T).select("_file").distinct().collect()}
new   = [p for p in files if p not in done]
print(f"{len(files)} dump files, {len(new)} new")

schema = StructType([StructField("station_id", LongType()),
                     StructField("csmd_json",  StringType()),
                     StructField("attr_json",  StringType())])

for path in new:
    snap = path.split("snapshot_date=")[1].split("/")[0]
    with open(path, encoding="utf-8") as f:
        dump = json.load(f)
    rows = [(s["csmd"]["id"],
             json.dumps(s["csmd"], ensure_ascii=False),
             json.dumps(s["attr"], ensure_ascii=False) if s.get("attr") is not None else None)
            for s in dump["chargerstations"]]
    (spark.createDataFrame(rows, schema)
          .select("station_id",
                  F.expr("parse_json(csmd_json)").alias("csmd"),
                  F.expr("parse_json(attr_json)").alias("attr"),
                  F.lit(snap).cast("date").alias("_snapshot_date"),
                  F.lit(dump.get("Rights")).alias("_rights"),
                  F.lit(path).alias("_file"),
                  F.current_timestamp().alias("_ingested_at"))
          .write.mode("append").saveAsTable(T))
    print(f"{path}: {len(rows):,} rows")

Verification

In [0]:
%sql
-- 1. One row per station, and the shape of attr (PHP's [] quirk)
SELECT _snapshot_date, count(*) AS rows, count(DISTINCT station_id) AS ids,
       count_if(schema_of_variant(attr:st)   LIKE 'ARRAY%') AS st_arrays,
       count_if(schema_of_variant(attr:conn) LIKE 'ARRAY%') AS conn_arrays,
       count_if(attr IS NULL) AS no_attr
FROM laddstolpar_df.bronze.nobil_stations GROUP BY 1;

-- 2. Type profile of every csmd field across all stations (checks the finding 30 correction)
SELECT schema_of_variant_agg(csmd) FROM laddstolpar_df.bronze.nobil_stations;